In [0]:
from pyspark.sql import functions as F

transactions = spark.table(
    "workspace.pyspark_deep_dive.transactions"
)

print(transactions.count())

In [0]:
max_date = transactions.select(
    F.max("transaction_date").alias("max_date")
).first()["max_date"]

print("Latest date:", max_date)

In [0]:
incremental_batch = (
    spark.range(1_000_001, 1_050_001)
    .withColumnRenamed("id", "transaction_id")
    .withColumn(
        "customer_id",
        (F.rand(seed=100) * 100_000).cast("int") + 1
    )
    .withColumn(
        "product_id",
        (F.rand(seed=200) * 10_000).cast("int") + 1
    )
    .withColumn(
        "quantity",
        (F.rand(seed=300) * 5).cast("int") + 1
    )
    .withColumn(
        "unit_price",
        F.round(F.rand(seed=400) * 1000 + 50, 2)
    )
    .withColumn(
        "transaction_date",
        F.date_add(F.lit(max_date), 1)
    )
)

incremental_batch.show(5)
print("New records:", incremental_batch.count())

In [0]:
incremental_batch.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.pyspark_deep_dive.transactions_incremental_batch"
    )

In [0]:
transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.pyspark_deep_dive.transactions_target"
    )

target = spark.table(
    "workspace.pyspark_deep_dive.transactions_target"
)

print("Target count:", target.count())

In [0]:
%sql
MERGE INTO workspace.pyspark_deep_dive.transactions_target AS t
USING workspace.pyspark_deep_dive.transactions_incremental_batch AS s
ON t.transaction_id = s.transaction_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *

In [0]:
target = spark.table(
    "workspace.pyspark_deep_dive.transactions_target"
)

print("Target count:", target.count())

In [0]:
duplicate_batch = (
    incremental_batch
    .unionByName(
        incremental_batch.limit(100)
    )
)

print("Before dedup:", duplicate_batch.count())

In [0]:
deduped_batch = duplicate_batch.dropDuplicates(
    ["transaction_id"]
)

print("After dedup:", deduped_batch.count())

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

batch_with_updates = (
    duplicate_batch
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

In [0]:
window_spec = Window.partitionBy(
    "transaction_id"
).orderBy(
    F.col("updated_at").desc()
)

latest_batch = (
    batch_with_updates
    .withColumn(
        "rn",
        F.row_number().over(window_spec)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [0]:
print("Before:", batch_with_updates.count())
print("After:", latest_batch.count())

In [0]:
latest_batch.createOrReplaceTempView("latest_batch")

In [0]:
%sql
MERGE INTO workspace.pyspark_deep_dive.transactions_target AS t
USING latest_batch AS s
ON t.transaction_id = s.transaction_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *

In [0]:
target = spark.table(
    "workspace.pyspark_deep_dive.transactions_target"
)

print("Target count:", target.count())